# 14 · Grad-CAM 시각화 — 설명 가능한 AI ⭐

**평가 항목 4번 (시각화 및 해석, 15%) 핵심 보강**

**목적**: 학습된 Attention U-Net이 **어느 영역을 보고 판단**하는지 시각화.

**방법**: Grad-CAM (Selvaraju et al. 2017, ICCV)
- 마지막 encoder conv layer (ResNet-34의 layer4)에 forward/backward hook
- 각 클래스의 픽셀 평균 score → backward → activation × gradient channel-wise sum
- 입력 크기로 upsample → 원본에 heatmap overlay

**산출물**: 클래스별(코/눈/입/피부) Grad-CAM heatmap 시각화

**발표 활용**: "우리 모델은 코끝 영역의 어디를 보고 nose 클래스로 판단했는가" 설명

## 1. 환경 셋업 + 모델 로드

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    !pip install -q segmentation_models_pytorch opencv-python pyyaml

print(f'CWD: {os.getcwd()}')

In [ ]:
import torch
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from project.segmentation.unet import build_unet

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Attention U-Net 로드 (Phase 6 학습 결과)
MODEL_CKPT = '/content/drive/MyDrive/cv-final-ckpts/unet_attention_best.pt'

model = build_unet(
    num_classes=6,
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=3,
    attention_type='scse',
).to(device)

if os.path.exists(MODEL_CKPT):
    model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
    print(f'✅ Attention U-Net 로드: {MODEL_CKPT}')
else:
    print(f'⚠ ckpt 없음 — 초기 가중치 사용 (Grad-CAM 결과는 noise)')

model.eval()
print(f'\n모델 구조 (encoder 일부):')
print(f'  encoder.conv1 → layer1 → layer2 → layer3 → layer4 (target)')
print(f'  decoder → segmentation_head → (B, 6, H, W) output')

## 2. 입력 이미지 + 전처리

In [ ]:
SIZE = 256  # U-Net 학습 시 사용한 크기
OUTPUT_DIR = Path('outputs/gradcam')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드:')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'samples/original_before.png'

image_bgr = cv2.imread(IMG_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

# 모델 입력 정규화 ([0, 255] → [0, 1])
input_tensor = torch.from_numpy(image_rgb).float() / 255.0
input_tensor = input_tensor.permute(2, 0, 1).unsqueeze(0).to(device)
input_tensor.requires_grad_(True)
print(f'입력 tensor shape: {input_tensor.shape}')

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input Image')
plt.show()

## 3. Grad-CAM 적용

**Target layer**: `encoder.layer4` (ResNet-34의 마지막 conv block)
- 가장 추상적인 high-level 특징 (계층적 특징 추출의 정점)
- 평가 항목 2번 (CNN 계층적 특징) 시각적 증명

In [ ]:
from project.segmentation.gradcam import GradCAM, find_target_layer, overlay_heatmap

# 클래스 정의 (config.yaml 기준)
CLASSES = {
    0: ('background', '배경'),
    1: ('nose', '코'),
    2: ('eye', '눈'),
    3: ('mouth', '입'),
    4: ('skin', '피부'),
}

# Target layer 선택
target_layer = find_target_layer(model, 'encoder.layer4')
print(f'Target layer: {type(target_layer).__name__}')

# Grad-CAM 인스턴스
gcam = GradCAM(model, target_layer)

# 클래스별 heatmap 생성
heatmaps = {}
for cls_idx, (cls_en, cls_ko) in CLASSES.items():
    if cls_en == 'background':
        continue  # background는 의미 없음
    
    # input 재생성 (gradient 누적 방지)
    inp = torch.from_numpy(image_rgb).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0).to(device)
    inp.requires_grad_(True)
    
    cam = gcam(inp, target_class=cls_idx)
    heatmaps[cls_en] = (cam.numpy(), cls_ko)
    print(f'  ✅ {cls_en} ({cls_ko}): cam range [{cam.min():.3f}, {cam.max():.3f}]')

gcam.remove_hooks()
print('\n✅ Grad-CAM 생성 완료')

## 4. 클래스별 Heatmap 시각화 ⭐ (발표 핵심 슬라이드)

In [ ]:
# 4 클래스 × (Heatmap + Overlay)
n_classes = len(heatmaps)
fig, axes = plt.subplots(n_classes, 3, figsize=(15, 5*n_classes))

for i, (cls_en, (cam_np, cls_ko)) in enumerate(heatmaps.items()):
    overlay = overlay_heatmap(image_rgb, cam_np, alpha=0.5)
    
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'Input', fontsize=14)
    axes[i, 1].imshow(cam_np, cmap='jet')
    axes[i, 1].set_title(f'Grad-CAM · {cls_ko} ({cls_en})', fontsize=14)
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'Overlay (alpha=0.5)', fontsize=14)
    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
save_path = OUTPUT_DIR / 'gradcam_all_classes.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_path}')

# 개별 클래스 PNG도 저장 (슬라이드용)
for cls_en, (cam_np, cls_ko) in heatmaps.items():
    overlay = overlay_heatmap(image_rgb, cam_np, alpha=0.5)
    cv2.imwrite(
        str(OUTPUT_DIR / f'gradcam_{cls_en}_overlay.png'),
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR),
    )
    cam_uint8 = (cam_np * 255).astype(np.uint8)
    cam_color = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    cv2.imwrite(str(OUTPUT_DIR / f'gradcam_{cls_en}_heatmap.png'), cam_color)

print(f'개별 PNG: {len(list(OUTPUT_DIR.glob("*.png")))} 개')

## 5. 발표용 단일 비교 이미지 (4 클래스 가로 배치)

In [ ]:
# 1 input + 4 overlays (가로 5컬럼)
fig, axes = plt.subplots(1, n_classes + 1, figsize=(4*(n_classes+1), 4))
axes[0].imshow(image_rgb)
axes[0].set_title('Input', fontsize=14)
axes[0].axis('off')

for i, (cls_en, (cam_np, cls_ko)) in enumerate(heatmaps.items()):
    overlay = overlay_heatmap(image_rgb, cam_np, alpha=0.5)
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f'{cls_ko} ({cls_en})', fontsize=14)
    axes[i + 1].axis('off')

plt.suptitle('Grad-CAM · 모델이 보는 영역', fontsize=16, y=1.02)
plt.tight_layout()
save_combo = OUTPUT_DIR / 'gradcam_presentation.png'
plt.savefig(save_combo, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 발표용: {save_combo}')

## 6. Drive 백업 + 학술 해석

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -v {OUTPUT_DIR}/*.png {DRIVE_OUT}/ 2>&1 | tail -10
    print(f'\n✅ Drive 백업: {DRIVE_OUT}/')

---

## 📚 학술 해석 (발표 슬라이드용)

**Grad-CAM 결과 해석**:
- 각 클래스(코/눈/입/피부)의 Grad-CAM은 **모델이 해당 클래스 판단 시 주목하는 공간적 영역**.
- 예: "코" 클래스 Grad-CAM에서 코 영역 + 주변이 강하게 활성화 → 모델이 정확히 코를 인식

**계층적 특징 추출 증명** (평가 항목 2번):
- ResNet-34 encoder의 layer4는 가장 추상적 high-level feature
- Conv1 (저수준 edge) → Layer4 (고수준 semantic) 의 단계적 추출 
- Grad-CAM이 의미 있는 영역을 짚는다 = encoder가 계층적 특징 잘 학습

**Attention U-Net의 시각적 증명**:
- SCSE Attention은 모델이 "어디/무엇"을 강조할지 학습
- Grad-CAM은 그 결과를 **사후적으로** 시각화
- 두 메커니즘이 서로 보완 (Roy 2018 + Selvaraju 2017)

**참고문헌**:
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017).
  *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*. ICCV.
- Roy, A. G., Navab, N., & Wachinger, C. (2018).
  *Concurrent Spatial and Channel Squeeze & Excitation in Fully Convolutional Networks*. MICCAI.

**발표 멘트 예시**:
> "본 시각화는 **Grad-CAM(Selvaraju 2017 ICCV)**으로, Attention U-Net이 
> **각 클래스 판단 시 주목한 영역**을 보여줍니다. '코' 클래스에서 코 영역이 
> 강하게 활성화된 것은 ResNet-34 encoder의 **계층적 특징 추출**이 의미 있는 
> high-level semantic을 학습했음을 입증하며, 본 모델이 **설명 가능한 비전 AI** 
> 원칙을 따름을 보여줍니다."